# 02 Target Definition

Neste notebook vamos comparar definições alternativos de inadimplência e escolher a target final para modelagem.

## Abordagem

Neste notebook, validamos definições alternativas de inadimplência (`ever_30`, `ever_60`, `ever_90`) e escolhemos a métrica mais adequada para modelagem com base em consistência e balanço da população.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

from src.data_loader import load_historico_emprestimos
from src.target import build_contract_target, choose_target_definition

emprestimos = load_historico_emprestimos()
contract_target = build_contract_target()
completed = emprestimos.merge(contract_target, on='id_contrato', how='left')
display(completed[['status_contrato', 'ever_30', 'ever_60', 'ever_90']].head())

In [ ]:
summary = completed.groupby('status_contrato')[['ever_30', 'ever_60', 'ever_90']].mean().reset_index()
display(summary)

approved = completed[completed['status_contrato'] == 'Approved']
print('Approved population size:', approved.shape[0])
print('Approved ever_30:', approved['ever_30'].mean())
print('Approved ever_60:', approved['ever_60'].mean())
print('Approved ever_90:', approved['ever_90'].mean())

## Escolha da Target

A análise indica que a melhor definição de risco de crédito para este case é a presença de pelo menos uma parcela com atraso superior a 60 dias (`ever_60`).
Essa definição é consistente com práticas de risco de crédito, representa um evento de inadimplência material e mantém a base suficientemente equilibrada para modelagem.

In [ ]:
final_target = choose_target_definition(completed)
final_target = final_target[final_target['status_contrato'] == 'Approved']
final_target = final_target[final_target['ever_60'].notna()]
from pathlib import Path
output_path = Path('..') / 'data' / 'processed' / 'population_target.parquet'
output_path.parent.mkdir(parents=True, exist_ok=True)
final_target.to_parquet(output_path, index=False)
print('Saved target dataset with shape', final_target.shape)